## Imports

In [1]:
from qsopt import * 
import numpy as np
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

## Define experimental parameters

In [2]:
gm = 0.03 * 2 * np.pi

# Define custom physical constants
custom_constants = PhysicalConstants(
    chi = 0.5 * gm,                    # Dispersive coupling
    photon_cavity_coupling = gm,  # Photon-cavity coupling
    inverse_pulse_width = 0.1 * gm      # Inverse pulse width
)

# Define custom system dimensions
custom_dims = SystemDimensions(
    cavity_levels=2,
    qubit_levels=2,
    field_levels=2
)

# Define measurement protocol
custom_measurement = MeasurementProtocol(
    measurement_times = list(np.array([-5.0, 0.0, 5.0])/(0.1 * gm))
)

# Define initial state configuration (SINGLE_PHOTON)
initial_state = InitialStateConfig(
    state_type=InitialStateType.SINGLE_PHOTON
)

# Define noise configuration
noise_config = NoiseConfiguration(
    depolarizing = 0.001,  
    dephasing = 0.001,      
    relaxation = 0.001
)

# Create parameters with custom configuration
exp_parameters = ExperimentalParameters(
    physical_constants=custom_constants,
    system_dims=custom_dims,
    measurement=custom_measurement,
    initial_state=initial_state,
    noise_config=noise_config
)

print(exp_parameters)

SYSTEM DIMENSIONS
  Cavity levels:             2
  Qubit levels:              2
  Field levels:              2
  Total dimension:           8
  Status:               VALID
PHYSICAL CONSTANTS
  Chi:                    0.0942
  Photon cavity coupling: 0.1885
  Inverse pulse width:    0.0188
  Status:               VALID
MEASUREMENT PROTOCOL
  Number of measurements:      3
  Measurement times: [np.float64(-265.25823848649225), np.float64(0.0), np.float64(265.25823848649225)]
  Status:               VALID
INITIAL STATE
  Type:                 single_photon
NOISE MODEL
  Depolarizing rate:      0.0010
  Dephasing rate:         0.0010
  Relaxation rate:        0.0010
  Custom operators:     None
  Status:               VALID
SYSTEM STATUS
  Configuration:        VALID


## Define trainable parameters

In [3]:
parameters = TrainableParameters()
parameters.add_rotation_angles(['ry1', 'ry2'], [np.pi/2, -np.pi/2], optimizer=optax.sgd(0.5))

print(parameters)

Trainable Parameters: 2
  Rotation Angles:
    ry1: 1.5708 rad (90.00°)
    ry2: -1.5708 rad (-90.00°)


## Define experiment

In [4]:
experiment = SingleQubitExperiment(exp_parameters, parameters)

## Test Single Simulation

In [5]:
results = experiment.run_simulation()

print(results)

MODE: Single Simulation
  Current Parameters:
     ry1: 1.570796 rad (90.00°)
     ry2: -1.570796 rad (-90.00°)
  Detection Probabilities:
     P(with photon):    0.780486
     P(without photon): 0.536414
     Contrast:          0.244072


## Run Optimization

In [ ]:


# The experiment automatically uses its built-in callback (saves every epoch by default)
history = experiment.optimize(
    theta_init=[1.5, -1.3],
    num_steps=30,
    verbose=True,
    tolerance=1e-7
)

Configuration:
    Max iterations: 70
    Convergence tolerance: 1.00e-09
    Initial rotation parameters: ry1=0.900 rad, ry2=-0.700 rad
    Optimizer: GradientTransformationExtraArgs
Step  ry1         ry2         Contrast    Grad Norm
----------------------------------------------------------------------


0     0.900000    -0.700000   0.144358    1.87e-01    
10    1.304832    -1.225262   0.235077    7.12e-02    
10    1.304832    -1.225262   0.235077    7.12e-02    
20    1.461678    -1.379924   0.245295    2.15e-02    
20    1.461678    -1.379924   0.245295    2.15e-02    
30    1.510423    -1.425450   0.246230    6.57e-03    
30    1.510423    -1.425450   0.246230    6.57e-03    
40    1.525389    -1.439398   0.246318    2.03e-03    
40    1.525389    -1.439398   0.246318    2.03e-03    
50    1.530004    -1.443717   0.246326    6.27e-04    
50    1.530004    -1.443717   0.246326    6.27e-04    
60    1.531431    -1.445057   0.246327    1.94e-04    
60    1.531431    -1.445057   0.246327    1.94e-04    


In [7]:
print(history)

MODE: Optimization
     Total iterations: 70
     Best epoch:        70
     Converged: False
     Final gradient norm: 6.774649e-05
  Best Parameters:
     ry1: 1.531848 rad (87.77°)
     ry2: -1.445449 rad (-82.82°)
  Detection Probabilities:
     P(with photon):    0.766092
     P(without photon): 0.519765
     Contrast:          0.246327
